In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("processed/tomato_daily.csv")

In [3]:
df["date"] = pd.to_datetime(df["date"])


In [4]:
full_dates = pd.date_range(
    start=df["date"].min(),
    end=df["date"].max(),
    freq="D"
)

In [5]:
missing_dates = full_dates.difference(df["date"])

In [6]:
print(" DATE COVERAGE ")
print("Total calendar days:", len(full_dates))
print("Observed days:", len(df))
print("Missing days:", len(missing_dates))

print(
    "Coverage:",
    round(len(df) / len(full_dates) * 100, 2),
    "%"
)

print("\nFirst 30 missing dates:")
print(missing_dates[:30])

 DATE COVERAGE 
Total calendar days: 2129
Observed days: 1849
Missing days: 280
Coverage: 86.85 %

First 30 missing dates:
DatetimeIndex(['2020-02-12', '2020-03-10', '2020-03-22', '2020-03-24',
               '2020-03-25', '2020-03-26', '2020-03-27', '2020-03-28',
               '2020-03-29', '2020-03-30', '2020-04-06', '2020-04-22',
               '2020-07-08', '2020-07-11', '2020-07-12', '2020-07-26',
               '2020-08-01', '2020-08-22', '2020-08-29', '2020-09-06',
               '2020-09-16', '2020-10-01', '2020-10-21', '2020-11-01',
               '2020-11-11', '2020-11-13', '2020-11-14', '2021-02-10',
               '2021-02-17', '2021-02-24'],
              dtype='datetime64[ns]', freq=None)


In [7]:
missing_series = pd.Series(missing_dates)

if len(missing_series) > 0:

    gap_groups = (
        missing_series
        .diff()
        .ne(pd.Timedelta(days=1))
        .cumsum()
    )

    gaps = (
        missing_series
        .groupby(gap_groups)
        .agg(["min", "max", "count"])
    )

    gaps = gaps.sort_values("count", ascending=False)

    print("\nMISSING GAPS ")
    print(gaps.head(20))


MISSING GAPS 
           min        max  count
116 2023-07-31 2023-08-13     14
168 2024-12-14 2024-12-23     10
4   2020-03-24 2020-03-30      7
165 2024-11-24 2024-11-29      6
119 2023-08-26 2023-08-30      5
171 2025-01-12 2025-01-16      5
88  2023-01-03 2023-01-06      4
153 2024-10-11 2024-10-14      4
77  2022-11-30 2022-12-02      3
131 2023-11-12 2023-11-14      3
152 2024-09-30 2024-10-02      3
145 2024-08-01 2024-08-03      3
167 2024-12-07 2024-12-09      3
170 2025-01-08 2025-01-10      3
172 2025-01-18 2025-01-20      3
73  2022-11-01 2022-11-03      3
115 2023-07-19 2023-07-20      2
66  2022-07-01 2022-07-02      2
59  2022-03-17 2022-03-18      2
80  2022-12-10 2022-12-11      2


In [8]:
print("\nPRICE STATISTICS")

print(df["price"].describe())


PRICE STATISTICS
count    1849.000000
mean     1835.437804
std      1018.371825
min       520.000000
25%      1150.000000
50%      1525.000000
75%      2300.000000
max      7450.000000
Name: price, dtype: float64


In [9]:
print("\nZero prices:", (df["price"] == 0).sum())
print("Negative prices:", (df["price"] < 0).sum())


Zero prices: 0
Negative prices: 0


In [10]:
print("\nDuplicate dates:", df["date"].duplicated().sum())



Duplicate dates: 0


In [11]:
df["year"] = df["date"].dt.year

yearly = (
    df.groupby("year")
      .agg(
          observations=("price", "count"),
          avg_price=("price", "mean"),
          min_price=("price", "min"),
          max_price=("price", "max")
      )
)

print("\n YEARLY SUMMARY ")
print(yearly)


 YEARLY SUMMARY 
      observations    avg_price  min_price  max_price
year                                                 
2020           339  1781.752212      920.0     3050.0
2021           327  1515.922018      520.0     3325.0
2022           324  1835.026235     1200.0     3035.0
2023           290  1876.232759      800.0     7450.0
2024           301  2295.789037      792.5     5650.0
2025           268  1732.520522      575.0     4000.0


In [14]:
df["date"] = pd.to_datetime(df["date"])

print(
    df.nlargest(20, "price")[["date", "price"]]
)

           date   price
1152 2023-07-16  7450.0
1149 2023-07-13  7430.0
1150 2023-07-14  7430.0
1153 2023-07-17  7350.0
1151 2023-07-15  7250.0
1164 2023-07-30  7100.0
1154 2023-07-18  7050.0
1162 2023-07-28  6900.0
1163 2023-07-29  6900.0
1161 2023-07-27  6800.0
1148 2023-07-12  6580.0
1146 2023-07-10  6530.0
1147 2023-07-11  6500.0
1160 2023-07-26  6325.0
1145 2023-07-09  6300.0
1143 2023-07-06  6200.0
1144 2023-07-07  6200.0
1159 2023-07-25  6050.0
1155 2023-07-21  6000.0
1165 2023-08-14  6000.0


In [15]:
print(
    df.nsmallest(20, "price")[["date", "price"]]
)

           date  price
479  2021-06-13  520.0
474  2021-06-08  550.0
477  2021-06-11  550.0
472  2021-06-06  560.0
451  2021-05-15  570.0
1659 2025-04-03  575.0
450  2021-05-14  580.0
473  2021-06-07  580.0
478  2021-06-12  580.0
449  2021-05-13  585.0
469  2021-06-03  585.0
462  2021-05-27  590.0
452  2021-05-17  600.0
456  2021-05-21  600.0
457  2021-05-22  600.0
458  2021-05-23  600.0
460  2021-05-25  600.0
463  2021-05-28  600.0
465  2021-05-30  600.0
467  2021-06-01  600.0
